# Soup GRPO task: step 1, baseline pilot on Qwen3-4B (Kaggle T4)

**Before Run All:** Settings, then **Accelerator: GPU T4 x2** (or T4) and **Internet: On**.
Takes about **2 hours**. When it finishes, download **`/kaggle/working/soup_results.zip`**
(Output panel, right side) and give it back.

What it does, in order (every step writes a timestamped log):
1. Unpacks the code and data embedded below and checks the data hashes.
2. Records the environment (GPU, driver, PCIe, RAM, torch).
3. Starts an nvidia-smi log (every second) and a **ballast process that holds everything above 8,192 MiB**,
   so this run physically cannot use more than an 8 GB card would (T4 has 16 GB).
4. Unit tests and **CUDA equivalence** of the streaming engine (prefetch on, 2 and 3 buffers).
5. **PCIe bandwidth** benchmark.
6. **fp16 check (H30):** the T4 has no bf16, so fp16 is compared with fp32 on the real Qwen3-4B.
7. **Prompt A/B (H31) and pilot (H24):** base Qwen3-4B, 100 puzzles × 4 answers. First both prompt versions at
   4 numbers, then the chosen prompt at 3 and 5 numbers (more levels only if the pre-registered rule says so),
   then the summary that picks the training level.

In [ ]:
# 1. Unpack code + data (git 4d22b00+dirty, bundle sha256 308fb6e390f5a8d5)
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
from pathlib import Path
LOCAL_TEST = os.environ.get("SOUP_LOCAL_TEST") == "1"   # only for testing this notebook on a laptop
WORK = Path(os.environ.get("SOUP_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = "..."
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == "308fb6e390f5a8d5e03bbc4af2e9ee056889eef1084a42393b72b4bd3df7ef57", "bundle corrupted"
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(WORK)
REPO = WORK / "soup-grpo"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
man = json.loads((REPO / "data/MANIFEST.json").read_text())
for f, info in man["files"].items():
    assert hashlib.sha256((REPO / "data" / f).read_bytes()).hexdigest() == info["sha256"], f
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
(REPO / "results/pilot").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| data files verified:", len(man["files"]), "| bundle git rev 4d22b00+dirty")

In [ ]:
# helper: run a command, stream its output live, keep a copy in results/logs
def run(cmd, log=None, check=True, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    t0 = time.time()
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    lines = []
    for line in p.stdout:
        if "Warning" in line or "warn(" in line:
            continue
        print(line, end="", flush=True)
        lines.append(line)
    p.wait()
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[exit {p.returncode}, {time.time() - t0:.0f}s]", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p.returncode

PY = sys.executable
STOP = None   # set by the first failing step; later steps are skipped, results are still packed
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU, as in the task
os.environ["TOKENIZERS_PARALLELISM"] = "false"
MODEL = "Qwen/Qwen3-0.6B" if LOCAL_TEST else "Qwen/Qwen3-4B"
DEVICE = "cpu" if LOCAL_TEST else "cuda"

In [ ]:
# 2. Environment + dependencies
if not LOCAL_TEST:
    run([PY, "-m", "pip", "install", "-q", "transformers>=4.51", "safetensors>=0.4", "huggingface_hub"], check=False)
    TS = time.strftime("%Y%m%d-%H%M%S")
    env_txt = subprocess.run("nvidia-smi; nvidia-smi --query-gpu=name,memory.total,driver_version,"
                             "pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max "
                             "--format=csv; free -g; nproc; df -h /kaggle/working",
                             shell=True, capture_output=True, text=True).stdout
    import torch, transformers
    env_txt += f"\ntorch {torch.__version__} cuda {torch.version.cuda} transformers {transformers.__version__}"
    env_txt += f"\ndevice {torch.cuda.get_device_name(0)} capability {torch.cuda.get_device_capability(0)}"
    env_txt += f" bf16_supported {torch.cuda.is_bf16_supported()}"
    (REPO / f"results/logs/env_{TS}.txt").write_text(env_txt)
    print(env_txt)

In [ ]:
# 3. Session monitor (nvidia-smi + host RAM every second) and the 8 GB ballast
if STOP is None:
    try:
        if not LOCAL_TEST:
            TS = time.strftime("%Y%m%d-%H%M%S")
            smi_log = open(REPO / f"results/logs/nvsmi_{TS}.csv", "w")
            monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,memory.used,memory.total,utilization.gpu,"
                                        "pcie.link.gen.current,pcie.link.width.current,temperature.gpu,power.draw,clocks.sm",
                                        "--format=csv", "-l", "1"], stdout=smi_log, stderr=subprocess.STDOUT)
            host_mon = subprocess.Popen(["bash", "-c", f"while true; do echo \"$(date -Is),$(free -m | awk '/Mem:/ {{print $3\",\"$7}}')\""
                                         f" >> {REPO}/results/logs/hostmem_{TS}.csv; sleep 1; done"])
            ballast = subprocess.Popen([PY, "scripts/gpu/ballast.py", "--cap-mib", "8192"],
                                       stdout=open(REPO / "results/logs/ballast.out", "w"), stderr=subprocess.STDOUT)
            for _ in range(120):
                if (REPO / "results/logs/ballast.json").exists():
                    break
                time.sleep(1)
            BALLAST = json.loads((REPO / "results/logs/ballast.json").read_text())
            print("ballast:", BALLAST)
            assert BALLAST["free_left_mib"] <= 8192, "cap not in place"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 4. Model download (~8 GB) and correctness of the engine on this GPU
if STOP is None:
    try:
        t0 = time.time()
        from soupgrpo.runtime import resolve_model_dir
        print("model at", resolve_model_dir(MODEL), f"({time.time() - t0:.0f}s)")
        run([PY, "-m", "pytest", "-q", "-x"], log="pytest.txt", check=False)
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/cuda_equivalence.py"], log="cuda_equivalence.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 5. PCIe bandwidth (pinned vs pageable, one 4B layer = 202 MB) + CUDA context cost
if STOP is None:
    try:
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/pcie_bench.py"], log="pcie.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 6. H30: fp16 vs fp32 on the real model (T4 has no bf16). Stop here if it fails.
if STOP is None:
    try:
        n, mx = (3, 32) if LOCAL_TEST else (8, 128)
        run([PY, "scripts/gpu/fp16_check.py", "--model", MODEL, "--device", DEVICE, "--n", n, "--max-new", mx],
            log="fp16_check.txt")
        H30 = json.loads(sorted((REPO / "results/logs").glob("fp16_check_*.json"))[-1].read_text())
        print("H30 PASS:", H30["PASS"])
        assert H30["PASS"], "fp16 changes the model's behaviour: do not pilot in fp16 (see results/logs)"
        if os.environ.get("SOUP_FORCE_FAIL") == "fp16":   # local test of the failure path only
            raise AssertionError("forced failure (test)")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 7. Pilot (H24) + prompt A/B (H31): base model, 100 puzzles x 4 answers, T=1.0, 256 new tokens, fp16
if STOP is None:
    try:
        DTYPE = "fp32" if LOCAL_TEST else "fp16"
        LIMIT = ["--limit", "6"] if LOCAL_TEST else []
        MAXNEW = "48" if LOCAL_TEST else "256"
        def pilot(level, prompt):
            out = REPO / f"results/pilot/{level}_{prompt}"
            run([PY, "scripts/sample_eval.py", "--model", MODEL, "--device", DEVICE, "--dtype", DTYPE, "--prompt", prompt,
                 "--data", f"data/pilot_{level}.jsonl", "--samples", "4", "--temperature", "1.0",
                 "--max-new", MAXNEW, "--batch", "64", "--out", out, *LIMIT], log=f"pilot_{level}_{prompt}.txt")
            return json.loads(out.with_suffix(".summary.json").read_text())

        # H31 prompt A/B at the middle level (rule pre-registered in predictions/PREDICTIONS.md)
        ab = {p: pilot("n4", p) for p in ("v1", "v2")}
        def in_range(s): return 0.20 <= s["accuracy"] <= 0.60
        if in_range(ab["v1"]) != in_range(ab["v2"]):
            PROMPT = "v1" if in_range(ab["v1"]) else "v2"
        elif in_range(ab["v1"]):
            PROMPT = "v1" if ab["v1"]["mixed_prompt_share"] > ab["v2"]["mixed_prompt_share"] else "v2"
        else:
            PROMPT = "v1" if ab["v1"]["accuracy"] > ab["v2"]["accuracy"] else "v2"
        print({p: round(s["accuracy"], 3) for p, s in ab.items()}, "-> prompt", PROMPT)

        acc = {"n4": ab[PROMPT]["accuracy"]}
        for level in ["n3", "n5"]:              # both neighbours of the middle level (pre-registered order)
            acc[level] = pilot(level, PROMPT)["accuracy"]
            print(f"==> {level}: {acc[level]:.1%}")
        # search loop from the plan: only if no level landed in 20-60%
        if not any(0.20 <= a <= 0.60 for a in acc.values()):
            if all(a >= 0.90 for a in acc.values()):
                acc["n6"] = pilot("n6", PROMPT)["accuracy"]                # harder: 6 numbers, targets 100-1000
            elif all(a <= 0.20 for a in acc.values()):
                acc["n3_addsub"] = pilot("n3_addsub", PROMPT)["accuracy"]  # easier: only + and -
        (REPO / "results/pilot/choice.json").write_text(json.dumps({"prompt": PROMPT, "accuracy": acc}, indent=2))
        print(acc)
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 8. Summary with the pre-registered selection rule + answers to read by hand
if STOP is None:
    try:
        run([PY, "scripts/pilot_summary.py", *sorted(map(str, (REPO / "results/pilot").glob("*.summary.json")))],
            log="pilot_summary.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Peak memory of THIS run (nvidia-smi used minus the ballast) and pack results (always runs)
if not LOCAL_TEST and "smi_log" in globals() and "BALLAST" in globals():
    import csv
    rows = list(csv.reader(open(smi_log.name)))[1:]
    used = [int(r[1].strip().split()[0]) for r in rows if len(r) > 1 and r[1].strip().split()[0].isdigit()]
    peak = max(used) - BALLAST["ballast_used_mib"]
    cap = {"smi_peak_total_mib": max(used), "ballast_used_mib": BALLAST["ballast_used_mib"],
           "our_peak_mib": peak, "under_8192": peak < 8192, "samples": len(used)}
    (REPO / "results/logs/cap_summary.json").write_text(json.dumps(cap, indent=2))
    print(cap)
import shutil
(REPO / "results/logs/stop_reason.txt").write_text(str(STOP))
print("STOP reason:", STOP)
zip_base = WORK / "soup_results"
shutil.make_archive(str(zip_base), "zip", REPO / "results")
print("DOWNLOAD:", str(zip_base) + ".zip")